In [9]:
import cv2

In [10]:
def track_object(video_path):
    cap = cv2.VideoCapture(video_path)

    # Step b: Create background subtractor
    background_subtractor = cv2.createBackgroundSubtractorMOG2()

    # Initialize variables for object tracking
    object_id = 0
    objects = {}  # Dictionary to store centroids of objects

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Step b: Apply background subtraction
        fg_mask = background_subtractor.apply(frame)

        # Apply noise filtering
        fg_mask = cv2.erode(fg_mask, None, iterations=2)
        fg_mask = cv2.dilate(fg_mask, None, iterations=2)

        # Step c: Find contours
        contours, _ = cv2.findContours(
            fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        # Step d: Determine centroid and assign unique object ids
        for contour in contours:
            area = cv2.contourArea(contour)
            if area > 1000:  # Adjust threshold according to your needs
                x, y, w, h = cv2.boundingRect(contour)
                centroid = (int(x + w / 2), int(y + h / 2))

                # Check if the object is new or existing
                new_object = True
                for obj_id, obj_centroid in objects.items():
                    dist = (
                        (centroid[0] - obj_centroid[0]) ** 2
                        + (centroid[1] - obj_centroid[1]) ** 2
                    ) ** 0.5
                    if dist < 50:  # Adjust distance threshold for object association
                        objects[obj_id] = centroid
                        new_object = False
                        break

                if new_object:
                    object_id += 1
                    objects[object_id] = centroid

        # Step e: Draw bounding boxes and object ids
        for obj_id, centroid in objects.items():
            cv2.putText(
                frame,
                f"ID: {obj_id}",
                (centroid[0] - 10, centroid[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2,
            )
            cv2.circle(frame, centroid, 5, (0, 255, 0), -1)

            # Check if object has moved out of frame and delete its ID
            if (
                centroid[0] <= 0
                or centroid[0] >= frame.shape[1]
                or centroid[1] <= 0
                or centroid[1] >= frame.shape[0]
            ):
                del objects[obj_id]

        cv2.imshow("Frame", frame)

        if cv2.waitKey(30) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

In [13]:
# Step a:
video_path = "road_video.mp4"
track_object(video_path)